In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
import os


train_dir = "Dataset/train"  
validation_dir = "Dataset/validation"  


train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),  
    batch_size=64,         
    label_mode='int',      
)

validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(224, 224),  
    batch_size=64,
    label_mode='int',
)


data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])


train_dataset = train_dataset.map(lambda x, y: (data_augmentation(x), y))


In [ ]:

def create_cnn_model(K):  
    model = tf.keras.Sequential([
        # Convolutional block 1
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu", input_shape=(224, 224, 3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        
        # Convolutional block 2
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        
        # Convolutional block 3
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        
        # Convolutional block 4
        tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2),
        
        # Flatten and Dense layers
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1024, activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(K, activation="softmax")  
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',  
        metrics=['accuracy']
    )
    
    return model


In [ ]:

K = 39
model = create_cnn_model(K)

epochs = 20

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)


In [ ]:
validation_loss, validation_accuracy = model.evaluate(validation_dataset)
print(f"Validation Accuracy: {validation_accuracy*100:.2f}%")

# Save the trained model
model.save("plant_disease_model_tf.h5")